In [53]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
import sys
sys.path.append(os.path.abspath('..'))

import pickle
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from scipy.stats import f

from config import get_config
from analysis.util import get_Xy_cbapm, get_Xy_real_concept

In [38]:
def grs_test(returns: pd.DataFrame,
             factors: pd.DataFrame,
             rf: pd.Series | float | None = None,
             asset_names: list[str] | None = None):
    """
    GRS test for pricing errors (alphas) of a linear factor model.

    Parameters
    ----------
    returns : pd.DataFrame (T x N)
        Asset returns (level returns). Index should align with `factors`.
    factors : pd.DataFrame (T x K)
        Factor returns (e.g., MKT, SMB, HML, RMW, CMA, UMD, etc.).
    rf : pd.Series | float | None
        Risk-free rate (same index as returns/factors). If None, treated as 0.
    asset_names : list[str] | None
        Optional names for assets; defaults to returns.columns.

    Returns
    -------
    result : dict
        {
          'F': GRS F-statistic,
          'pval': p-value,
          'alpha_monthly': np.ndarray (N,),
          'alpha_annual': np.ndarray (N,),
          'alpha_mean_abs_monthly': float,
          'alpha_mean_abs_annual': float,
          'alpha_rms_monthly': float,
          'alpha_rms_annual': float,
          'alpha_df': pd.DataFrame (per-asset alphas),
          'S': np.ndarray residual covariance (N x N),
          'mu_f': np.ndarray (K,),
          'Sigma_f': np.ndarray (K x K),
          'T': int, 'N': int, 'K': int
        }
    """

    # 0) Align indices and basic shapes
    R = returns.copy()
    F = factors.copy()
    if rf is None:
        rf_vec = np.zeros(len(R))
    elif np.isscalar(rf):
        rf_vec = np.full(len(R), float(rf))
    else:
        rf_vec = rf.loc[R.index].values

    # Align factors to returns index
    F = F.loc[R.index]

    # Drop any rows with NA across R/F/rf
    df_all = pd.concat([R, F, pd.Series(rf_vec, index=R.index, name='_rf_')], axis=1)
    df_all = df_all.dropna()
    T = len(df_all)
    if T == 0:
        raise ValueError("No overlapping, non-missing observations after alignment.")

    # Re-split
    R = df_all[R.columns]
    F = df_all[F.columns]
    rf_vec = df_all['_rf_'].values

    # 1) Excess returns
    Rex = R.values - rf_vec[:, None]          # T x N
    X = np.column_stack([np.ones(T), F.values])  # T x (1+K)
    N = Rex.shape[1]
    K = F.shape[1]

    # Degree of freedom check
    if T <= (K + 1):
        raise ValueError(f"Insufficient T relative to K: need T > K+1, got T={T}, K={K}")
    if T <= (N + K):
        raise ValueError(f"Insufficient T relative to N and K: need T > N+K, got T={T}, N={N}, K={K}")

    # 2) OLS per asset: Rex = a + B F + e
    XtX = X.T @ X
    try:
        XtX_inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(XtX)  # If XtX is singular, use pseudo-inverse

    beta_hat = XtX_inv @ (X.T @ Rex)  # (1+K) x N
    alpha = beta_hat[0, :]            # N-vector (monthly alphas)
    resid = Rex - X @ beta_hat        # T x N

    # Residual covariance matrix S (unbiased)
    S = (resid.T @ resid) / (T - (K + 1))   # N x N
    try:
        S_inv = np.linalg.inv(S)
    except np.linalg.LinAlgError:
        S_inv = np.linalg.pinv(S)           # If S is singular, use pseudo-inverse

    # 3) Factor moments
    mu_f = F.mean().values.reshape(-1, 1)    # K x 1
    Sigma_f = np.cov(F.values, rowvar=False, ddof=1)  # K x K or scalar (K=1)
    Sigma_f = np.atleast_2d(Sigma_f)
    try:
        Sigma_f_inv = np.linalg.inv(Sigma_f)
    except np.linalg.LinAlgError:
        Sigma_f_inv = np.linalg.pinv(Sigma_f)  # If Sigma_f is singular, use pseudo-inverse

    # 4) GRS statistic
    alpha_vec = alpha.reshape(-1, 1)         # N x 1

    term1 = float(alpha_vec.T @ S_inv @ alpha_vec)
    term2 = float(1 + (mu_f.T @ Sigma_f_inv @ mu_f))
    df_num = N
    df_den = T - N - K
    if df_den <= 0:
        raise ValueError(f"Nonpositive denominator dof: T-N-K = {df_den}. Reduce N or K, or extend T.")
    F_num = df_den / df_num
    F_stat = F_num * (term1 / term2)
    pval = 1 - f.cdf(F_stat, df_num, df_den)

    # 5) Alpha summaries (monthly)
    alpha_mean_abs = float(np.mean(np.abs(alpha)))
    alpha_rms = float(np.sqrt(np.mean(alpha ** 2)))

    # 6) Annualize alphas (compounded)
    alpha_annual = (1 + alpha)**12 - 1
    alpha_mean_abs_annual = float(np.mean(np.abs(alpha_annual)))
    alpha_rms_annual = float(np.sqrt(np.mean(alpha_annual ** 2)))

    # Alpha DataFrame
    if asset_names is None:
        asset_names = list(R.columns)
    alpha_df = pd.DataFrame({
        'alpha_monthly': alpha,
        'alpha_annual': alpha_annual
    }, index=asset_names)

    return {
        'F': F_stat,
        'pval': pval,
        'alpha_monthly': alpha,
        'alpha_annual': alpha_annual,
        'alpha_mean_abs_monthly': alpha_mean_abs,
        'alpha_mean_abs_annual': alpha_mean_abs_annual,
        'alpha_rms_monthly': alpha_rms,
        'alpha_rms_annual': alpha_rms_annual,
        'alpha_df': alpha_df,
        'S': S,
        'mu_f': mu_f.flatten(),
        'Sigma_f': Sigma_f,
        'T': T, 'N': N, 'K': K
    }

In [96]:
def _to_datetime_index(df: pd.DataFrame, date_col: str = 'date', fmt: str | None = '%Y%m') -> pd.DataFrame:
    """
    Ensure DatetimeIndex from a date column.
    If fmt is None, let pandas infer. Otherwise use the given strptime format (e.g., '%Y%m').
    """
    out = df.copy()
    if not np.issubdtype(out[date_col].dtype, np.datetime64):
        if fmt is None:
            out[date_col] = pd.to_datetime(out[date_col])
        else:
            out[date_col] = pd.to_datetime(out[date_col].astype(str).str.strip(), format=fmt)
    out = out.set_index(date_col)
    out.index.name = date_col
    return out


def _restrict_period(df, start_dt, end_dt):
    df = df.sort_index()
    return df.loc[(df.index >= start_dt) & (df.index <= end_dt)]

def _load_ff_like_csv(path, rename_map=None, percent_cols=None):
    """
    Read a Ken French-style CSV file, convert to DatetimeIndex (YYYY-MM-01, not month-end),
    and convert percentage values to decimals.
    """
    df = pd.read_csv(path, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    # Keep only YYYYMM (6 digits)
    mask_ym = df['date'].astype(str).str.strip().str.fullmatch(r"\d{6}")
    df = df.loc[mask_ym].copy()
    # Convert date column
    df['date'] = pd.to_datetime(df['date'].astype(str).str.strip(), format='%Y%m')
    df = df.set_index('date')
    # Rename columns if needed
    if rename_map:
        df = df.rename(columns=rename_map)
    # Convert % to decimals
    if percent_cols is None:
        percent_cols = df.columns.tolist()
    df[percent_cols] = df[percent_cols].astype(float) / 100.0
    return df

def _load_standard_factors(base_dir="../data"):
    """
    Load standard factor datasets:
      - FF3: "../data/F-F_Research_Data_Factors.csv"  (Mkt-RF, SMB, HML, RF)
      - FF5: "../data/F-F_Research_Data_5_Factors_2x3.csv" (Mkt-RF, SMB, HML, RMW, CMA, RF)
      - MOM: "../data/F-F_Momentum_Factor.csv" (UMD)
    Returns a dict with keys {'CAPM','FF3','Carhart4','FF5','FF6'} (only those available).
    """
    out = {}

    # FF3
    ff3_path = os.path.join(base_dir, "F-F_Research_Data_Factors.csv")
    ff3 = _load_ff_like_csv(ff3_path, rename_map={'Mkt-RF':'MKT'})
    # Split factors and RF
    ff3_factors = ff3[['MKT','SMB','HML']].copy()
    rf = ff3['RF'].copy()

    # CAPM
    out['CAPM'] = {'factors': ff3[['MKT']].copy(), 'rf': rf.copy()}

    # FF3
    out['FF3'] = {'factors': ff3_factors.copy(), 'rf': rf.copy()}

    # FF5
    ff5_path = os.path.join(base_dir, "F-F_Research_Data_5_Factors_2x3.csv")
    if os.path.exists(ff5_path):
        ff5 = _load_ff_like_csv(ff5_path, rename_map={'Mkt-RF':'MKT'})
        # Some distributions do not include RF; use FF3's RF as fallback
        rf5 = ff5['RF'] if 'RF' in ff5.columns else rf.reindex(ff5.index)
        out['FF5'] = {
            'factors': ff5[['MKT','SMB','HML','RMW','CMA']].copy(),
            'rf': rf5.copy()
        }

    # MOM (UMD)
    mom_path = os.path.join(base_dir, "F-F_Momentum_Factor.csv")
    if os.path.exists(mom_path):
        mom = _load_ff_like_csv(mom_path, rename_map={'Mom   ': 'UMD', 'Mom':'UMD'})
        # Carhart 4 = FF3 + UMD
        if 'FF3' in out:
            common_idx = out['FF3']['factors'].index.intersection(mom.index)
            carhart4 = pd.concat(
                [out['FF3']['factors'].loc[common_idx], mom[['UMD']].loc[common_idx]],
                axis=1
            )
            out['Carhart4'] = {'factors': carhart4, 'rf': out['FF3']['rf'].loc[common_idx]}

        # FF6 = FF5 + UMD
        if 'FF5' in out:
            common_idx = out['FF5']['factors'].index.intersection(mom.index)
            ff6 = pd.concat(
                [out['FF5']['factors'].loc[common_idx], mom[['UMD']].loc[common_idx]],
                axis=1
            )
            rf6 = out['FF5']['rf'].loc[common_idx]
            out['FF6'] = {'factors': ff6, 'rf': rf6}

    return out

def _run_grs_named(returns, model_name, factors_df, rf_series):
    res = grs_test(returns, factors_df, rf_series)
    pretty = {
        'Factor Model': model_name,
        'GRS F-statistic': res['F'],
        'p-value': res['pval'],
        'Mean Abs Alpha (Monthly)': res['alpha_mean_abs_monthly'],
        'Mean Abs Alpha (Annual)': res['alpha_mean_abs_annual'],
        'RMS Alpha (Monthly)': res['alpha_rms_monthly'],
        'RMS Alpha (Annual)': res['alpha_rms_annual'],
        'Number of Factors (K)': res['K'],
        'Sample Size (T)': res['T'],
        'Number of Assets (N)': res['N']
    }
    return pretty, res

In [97]:
def extract_model_factors_for_grs(
    horizon,
    weight_lambda,
    embedding_method='autoencoder',
    train_date='2020-01-01'
):
    """
    Construct CB-APM factor-mimicking portfolios (value-weighted long–short deciles)
    for GRS tests. Size (market equity) from input_df is used for VW weights.
    """

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # ---------------------------------------------------------------
    # 1. Load input (contains Size) and 1-month realized returns
    # ---------------------------------------------------------------
    input_df = pd.read_csv(f'../data/input_{horizon}.csv')
    target_df = pd.read_csv(f'../data/target_1month.csv')

    input_df['date'] = pd.to_datetime(input_df['date'])
    target_df['date'] = pd.to_datetime(target_df['date'])

    # Extract Size for VW weighting
    size_df = input_df[['date', 'permno', 'Size']].copy()

    # ---------------------------------------------------------------
    # 2. Load signal info and CB-APM configuration
    # ---------------------------------------------------------------
    signal_info = pd.read_csv('../data/info/SignalDoc.csv')
    info = signal_info[signal_info['Acronym'].isin(input_df.columns)]
    config = get_config(weight_lambda)

    # ---------------------------------------------------------------
    # 3. Compute CB-APM consensus (concept layer outputs)
    # ---------------------------------------------------------------
    X_cbapm_with_index, _ = get_Xy_cbapm(
        train_date=train_date,
        input=input_df,
        target=target_df,
        info=info,
        config=config,
        device=device,
        horizon=horizon,
        weight_lambda=weight_lambda,
        embedding_method=embedding_method
    )

    # ---------------------------------------------------------------
    # 4. Merge CBAPM → realized return → Size
    # ---------------------------------------------------------------
    df = (
        X_cbapm_with_index
        .merge(target_df[['date', 'permno', 'Return']], on=['date', 'permno'], how='inner')
        .merge(size_df, on=['date', 'permno'], how='inner')
    )

    # Restrict to pre-train window
    df = df[df['date'] < pd.to_datetime(train_date)].copy()
    df = df.sort_values(['date', 'permno'])

    # Identify concept columns (CNN/MLP consensus outputs)
    concept_cols = [
        c for c in df.columns
        if c not in ['date', 'permno', 'Return', 'Size']
    ]

    # ---------------------------------------------------------------
    # 5. Construct value-weighted long–short factor returns
    # ---------------------------------------------------------------
    factor_returns = []

    for concept in concept_cols:

        # rank into deciles each month
        df['rank'] = df.groupby('date')[concept].transform(
            lambda x: pd.qcut(x, 10, labels=False, duplicates='drop')
        )

        long_leg = df[df['rank'] == 9].copy()
        short_leg = df[df['rank'] == 0].copy()
        
        # VW long: sum(Size*Return) / sum(Size)
        long_ret = (
            long_leg.groupby('date').apply(lambda x: (x['Size'] * x['Return']).sum() / x['Size'].sum())
        )

        # VW short
        short_ret = (
            short_leg.groupby('date').apply(lambda x: (x['Size'] * x['Return']).sum() / x['Size'].sum())
        )

        # Long–short factor
        factor_series = long_ret - short_ret
        factor_series.name = concept

        factor_returns.append(factor_series)

    # ---------------------------------------------------------------
    # 6. Combine into a T × K factor return panel
    # ---------------------------------------------------------------
    factors_df = pd.concat(factor_returns, axis=1).sort_index()

    print(f"[INFO] CB-APM long–short factors constructed.")
    print(f"       shape={factors_df.shape}, K={len(concept_cols)}")
    print(f"       Concepts: {concept_cols[:5]}{'...' if len(concept_cols)>5 else ''}")

    return factors_df




def prepare_portfolio_returns_generic(source,
                                      date_col: str = 'date',
                                      yyyymm: bool = True,
                                      percent_to_decimal: bool = True,
                                      drop_non_numeric: bool = True,
                                      deduplicate: bool = True) -> pd.DataFrame:
    """
    Load/standardize arbitrary test-asset returns for the GRS test.

    Parameters
    ----------
    source : str | pd.DataFrame
        - CSV path (str), or
        - DataFrame already in memory (must contain `date_col`).
    date_col : str
        Column name that holds dates.
    yyyymm : bool
        If True, interpret date strings as 'YYYYMM' (Ken French style). If False, let pandas infer.
    percent_to_decimal : bool
        If True, divide all return columns by 100.
    drop_non_numeric : bool
        If True, coerce all columns to numeric and drop rows with any NaNs.
    deduplicate : bool
        If True, drop duplicate dates (keep the first).

    Returns
    -------
    returns_df : pd.DataFrame
        T x N matrix of returns with a monthly DatetimeIndex (YYYY-MM-01).
    """
    # 1) Load
    if isinstance(source, str):
        df = pd.read_csv(source, skipinitialspace=True)
    elif isinstance(source, pd.DataFrame):
        df = source.copy()
    else:
        raise TypeError("`source` must be a CSV path or a pandas DataFrame.")

    # 2) Strip column names
    df.columns = df.columns.str.strip()

    # 3) If the date column contains headers/footers (Ken French style), keep only YYYYMM rows
    if yyyymm:
        mask_ym = df[date_col].astype(str).str.strip().str.fullmatch(r"\d{6}")
        df = df.loc[mask_ym].copy()

    # 4) To DatetimeIndex
    fmt = '%Y%m' if yyyymm else None
    df = _to_datetime_index(df, date_col=date_col, fmt=fmt)

    # 5) Numeric clean-up
    if drop_non_numeric:
        df = df.apply(pd.to_numeric, errors='coerce')
        df = df.dropna(how='any')

    # 6) De-duplicate dates
    if deduplicate:
        df = df[~df.index.duplicated(keep='first')]

    # 7) Convert % → decimal
    if percent_to_decimal:
        df = df / 100.0

    return df

def run_grs_test_with_model_factors(
    horizon,
    weight_lambda,
    embedding_method='autoencoder',
    train_date='2020-01-01',
    start_date="1994-01-01",
    end_date="2023-11-30",
    portfolio_returns: pd.DataFrame | None = None,
    portfolio_csv_path: str | None = None,
    portfolio_date_col: str = 'date',
    portfolio_yyyymm: bool = True,
    portfolio_percent_to_decimal: bool = True
):
    """
    Run GRS test using model-inferred factors against a user-specified test portfolio.
    (explicitly provide CSV path or DataFrame.)
    """

    print(f"Running GRS test using model factors")
    print(f"Prediction horizon: {horizon}")
    print(f"Lambda: {weight_lambda}")
    print(f"Embedding method: {embedding_method}")
    print(f"Training date: {train_date}")
    print("-" * 50)

    # -----------------------------
    # 1. Load model factors
    # -----------------------------
    print("1. Extracting model factors time series...")
    model_factors = extract_model_factors_for_grs(horizon, weight_lambda, embedding_method, train_date)
    print(f"   Model factors shape: {model_factors.shape}")
    print(f"   Factor names: {list(model_factors.columns)[:5]}{'...' if model_factors.shape[1]>5 else ''}")

    # -----------------------------
    # 2. Load test portfolio (MUST be provided)
    # -----------------------------
    print("\n2. Preparing user-specified portfolio returns...")
    if portfolio_returns is not None:
        R = portfolio_returns.copy()
    elif portfolio_csv_path is not None:
        R = prepare_portfolio_returns_generic(
            portfolio_csv_path,
            date_col=portfolio_date_col,
            yyyymm=portfolio_yyyymm,
            percent_to_decimal=portfolio_percent_to_decimal
        )
    else:
        raise ValueError("You must provide either `portfolio_csv_path` or `portfolio_returns`.")

    print(f"   Portfolio returns shape: {R.shape}")

    # -----------------------------
    # 3. Align and restrict period
    # -----------------------------
    model_factors = model_factors.sort_index()
    R = R.sort_index()
    start_dt, end_dt = pd.to_datetime(start_date), pd.to_datetime(end_date)
    common_start = max(model_factors.index.min(), R.index.min(), start_dt)
    common_end = min(model_factors.index.max(), R.index.max(), end_dt)

    model_factors = model_factors.loc[(model_factors.index >= common_start) & (model_factors.index <= common_end)]
    R = R.loc[(R.index >= common_start) & (R.index <= common_end)]
    print(f"   Common period: {common_start.date()} → {common_end.date()}")
    print(f"   Final factors: {model_factors.shape} | Final returns: {R.shape}")

    # -----------------------------
    # 4. Run GRS test
    # -----------------------------
    print("\n4. Running GRS test...")
    result = grs_test(R, model_factors, rf=None)

    return result


def compare_model_vs_ff_factors(
    horizon,
    weight_lambda,  # float or list[float]
    embedding_method='autoencoder',
    train_date='2020-01-01',
    start_date="1994-01-01",
    end_date='2020-01-01',
    data_dir="../data",
    portfolio_returns: pd.DataFrame | None = None,
    portfolio_csv_path: str | None = None,
    portfolio_date_col: str = 'date',
    portfolio_yyyymm: bool = True,
    portfolio_percent_to_decimal: bool = True
):
    """
    Compare GRS results of model-inferred factors (for one or multiple lambdas)
    vs standard factor models (CAPM, FF3, Carhart4, FF5, FF6).

    `weight_lambda` can be a single float or a list of floats.
    """
    print("="*70)
    print("Model Factors (one or many lambdas) vs Standard Factor Models")
    print("="*70)

    # --- normalize weight_lambda to a list ---
    if isinstance(weight_lambda, (int, float)):
        lambda_list = [float(weight_lambda)]
    else:
        # assume iterable of floats
        lambda_list = [float(x) for x in weight_lambda]

    # Require explicit portfolio input
    if portfolio_returns is None and portfolio_csv_path is None:
        raise ValueError("You must provide either `portfolio_csv_path` or `portfolio_returns`.")

    start_dt, end_dt = pd.to_datetime(start_date), pd.to_datetime(end_date)

    # Load test portfolio once
    if portfolio_returns is not None:
        R_all = portfolio_returns.copy()
    else:
        R_all = prepare_portfolio_returns_generic(
            portfolio_csv_path,
            date_col=portfolio_date_col,
            yyyymm=portfolio_yyyymm,
            percent_to_decimal=portfolio_percent_to_decimal
        )
    R_all = _restrict_period(R_all, start_dt, end_dt)

    # --- run GRS for each lambda ---
    model_rows = []
    details = {}
    for lam in lambda_list:
        label = f"Model Factor (λ={lam:g})"
        print(f"\n[Model] Running GRS for {label}")
        model_res = run_grs_test_with_model_factors(
            horizon=horizon,
            weight_lambda=lam,
            embedding_method=embedding_method,
            train_date=train_date,
            start_date=start_date,
            end_date=end_date,
            portfolio_returns=R_all  # reuse the same test assets
        )
        details[label] = model_res
        model_rows.append({
            'Factor Model': label,
            'GRS F-statistic': model_res['F'],
            'p-value': model_res['pval'],
            'Mean Abs Alpha (Monthly)': model_res['alpha_mean_abs_monthly'],
            'Mean Abs Alpha (Annual)': model_res['alpha_mean_abs_annual'],
            'RMS Alpha (Monthly)': model_res['alpha_rms_monthly'],
            'RMS Alpha (Annual)': model_res['alpha_rms_annual'],
            'Number of Factors (K)': model_res['K'],
            'Sample Size (T)': model_res['T'],
            'Number of Assets (N)': model_res['N'],
        })

    # --- standard factor sets (compute once) ---
    std = _load_standard_factors(base_dir=data_dir)

    def _to_row(name: str, res: dict) -> dict:
        return {
            'Factor Model': name,
            'GRS F-statistic': res['F'],
            'p-value': res['pval'],
            'Mean Abs Alpha (Monthly)': res['alpha_mean_abs_monthly'],
            'Mean Abs Alpha (Annual)': res['alpha_mean_abs_annual'],
            'RMS Alpha (Monthly)': res['alpha_rms_monthly'],
            'RMS Alpha (Annual)': res['alpha_rms_annual'],
            'Number of Factors (K)': res['K'],
            'Sample Size (T)': res['T'],
            'Number of Assets (N)': res['N'],
        }

    std_rows = []
    for name in ['CAPM', 'FF3', 'Carhart4', 'FF5', 'FF6']:
        if name not in std:
            print(f"  - [Skip] {name}: Required file not found.")
            continue
        fac = _restrict_period(std[name]['factors'], start_dt, end_dt)
        rf = _restrict_period(std[name]['rf'], start_dt, end_dt)
        common_idx = R_all.index.intersection(fac.index).intersection(rf.index)
        if len(common_idx) < 24:
            print(f"  - [Skip] {name}: Too few overlapping observations (T={len(common_idx)}).")
            continue
        pretty, res = _run_grs_named(R_all.loc[common_idx], name, fac.loc[common_idx], rf.loc[common_idx])
        std_rows.append(pretty)
        details[name] = res

    # --- build one consolidated summary table ---
    comparison_df = pd.DataFrame(model_rows + std_rows)

    print("\n" + "="*70)
    print("Comparison Summary (same sample period)")
    print("="*70)
    print(comparison_df.to_string(index=False, float_format='%.6f'))

    return comparison_df, details


In [98]:
def compare_portfolio_vs_ff_factors(
    horizon: str = "12month",
    weight_lambdas: list[float] = [0.1, 0.5, 1.0],
    data_dir: str = "../data",
    results_dir: str = "../results",
    start_date: str = "1994-01-01",
    end_date: str = "2020-01-01",
):
    """
    Run GRS tests for CB-APM predicted portfolio returns (deciles)
    against standard Fama–French factor models.

    Each λ (training weight) corresponds to a decile-sorted portfolio
    constructed from CB-APM’s predicted returns.
    """
    print("="*70)
    print("CB-APM Predicted Decile Portfolios vs Standard Factor Models")
    print("="*70)

    start_dt, end_dt = pd.to_datetime(start_date), pd.to_datetime(end_date)

    # Load standard factor sets once
    std = _load_standard_factors(base_dir=data_dir)
    models_to_compare = ['CAPM', 'FF3', 'Carhart4', 'FF5', 'FF6']

    summary_rows = []
    details = {}

    for lam in weight_lambdas:
        file_path = os.path.join(results_dir, f"{horizon}_{lam}.pickle")
        if not os.path.exists(file_path):
            print(f"[Skip] {file_path} not found.")
            continue

        # -------------------------
        # 1. Load CB-APM results
        # -------------------------
        with open(file_path, "rb") as f:
            output = pickle.load(f)

        # CB-APM ex-ante expected return
        forecast = output.get("forecast_target")
        if forecast is None:
            print(f"[Skip] λ={lam} — no forecast_target key in pickle.")
            continue

        forecast.columns = ['date', 'permno', 'forecast']
        forecast['date'] = pd.to_datetime(forecast['date'])

        # ex-post realized return
        target_path = f"../data/target_1month.csv"
        if not os.path.exists(target_path):
            raise FileNotFoundError(f"Missing realized return file: {target_path}")
        target = pd.read_csv(target_path)
        target['date'] = pd.to_datetime(target['date'])
        target = target[['date', 'permno', 'Return']]  # assume column names
        target = target.rename(columns={'Return': 'actual'})

        # Merge to get (date, permno, forecast, actual)
        merged = pd.merge(forecast, target, on=['date', 'permno'], how='inner')
        merged = merged.dropna(subset=['forecast', 'actual'])

        # -------------------------
        # 2. Sort into deciles by forecast score
        # -------------------------
        decile_returns = []  # store per-date portfolio mean returns
        for date, df_date in merged.groupby('date'):
            if len(df_date) < 10:
                continue
            df_date = df_date.sort_values('forecast')
            df_date['decile'] = pd.qcut(df_date['forecast'], 10, labels=False) + 1  # 1~10
            decile_means = df_date.groupby('decile')['actual'].mean()
            decile_returns.append(pd.DataFrame({'date': date, **decile_means.to_dict()}, index=[0]))

        # Combine into DataFrame (T x 10)
        port = pd.concat(decile_returns, ignore_index=True)
        port = port.sort_values('date')
        port = port.set_index('date')

        # Rename columns for clarity
        port.columns = [f'Decile{int(c)}' for c in port.columns]
        port.index = pd.to_datetime(port.index)
        port = _restrict_period(port, start_dt, end_dt)

        # -------------------------
        # 3. Run GRS for each FF model
        # -------------------------
        for name in models_to_compare:
            if name not in std:
                continue
            fac = _restrict_period(std[name]['factors'], start_dt, end_dt)
            rf = _restrict_period(std[name]['rf'], start_dt, end_dt)

            common_idx = port.index.intersection(fac.index).intersection(rf.index)
            if len(common_idx) < 24:
                print(f"  - [Skip] {name}: T={len(common_idx)} too short.")
                continue

            R = port.loc[common_idx]
            F = fac.loc[common_idx]
            rf_use = rf.loc[common_idx]

            pretty, res = _run_grs_named(R, f"{name} (λ={lam})", F, rf_use)
            summary_rows.append(pretty)
            details[f"{name} (λ={lam})"] = res

    # -------------------------
    # 4. Summarize results
    # -------------------------
    if summary_rows:
        comparison_df = pd.DataFrame(summary_rows)
        print("\n" + "="*70)
        print("Summary of GRS Tests (Predicted Portfolios)")
        print("="*70)
        print(comparison_df.to_string(index=False, float_format='%.6f'))
    else:
        comparison_df = pd.DataFrame()
        print("No valid results produced.")

    return comparison_df, details


In [99]:
def compare_concept_portfolio_vs_ff_factors(
    horizon: str = "12month",
    weight_lambdas: list[float] = [0.1, 0.5, 1.0],
    data_dir: str = "../data",
    results_dir: str = "../results",
    start_date: str = "1994-01-01",
    end_date: str = "2020-01-01",
):
    """
    Run GRS tests for decile portfolios formed by all forecasted concept variables 
    (e.g., Analyst earnings per share, EPS Forecast Dispersion) from CB-APM 
    against standard Fama–French factor models.

    Each λ (training weight) corresponds to a decile-sorted portfolio
    constructed from CB-APM's forecasted concept values rather than predicted returns.
    """

    print("=" * 90)
    print("CB-APM Concept-Sorted Decile Portfolios vs Standard Factor Models")
    print("=" * 90)

    start_dt, end_dt = pd.to_datetime(start_date), pd.to_datetime(end_date)

    # Load standard Fama–French factors once
    std = _load_standard_factors(base_dir=data_dir)
    models_to_compare = ['CAPM', 'FF3', 'Carhart4', 'FF5', 'FF6']

    summary_rows = []
    details = {}

    # Iterate over λ values
    for lam in weight_lambdas:
        file_path = os.path.join(results_dir, f"{horizon}_{lam}.pickle")
        if not os.path.exists(file_path):
            print(f"[Skip] {file_path} not found.")
            continue

        # -------------------------
        # 1. Load CB-APM results
        # -------------------------
        with open(file_path, "rb") as f:
            output = pickle.load(f)

        cons = output.get("forecast_concept")
        if cons is None:
            print(f"[Skip] λ={lam} — no forecast_concept key in pickle.")
            continue

        # Standardize column names
        cons.columns = [
            'date', 'permno', 'EPS forecast revision', 'Change in recommendation',
            'Change in Forecast and Accrual', 'Long-vs-short EPS forecasts',
            'Analyst earnings per share', 'EPS Forecast Dispersion',
            'Earnings forecast revisions', 'Analyst Value', 'Analyst Optimism'
        ]

        concept_vars = cons.columns[2:]  # exclude date, permno

        # Load realized returns
        target_path = f"../data/target_1month.csv"
        if not os.path.exists(target_path):
            raise FileNotFoundError(f"Missing realized return file: {target_path}")
        target = pd.read_csv(target_path)
        target["date"] = pd.to_datetime(target["date"])
        target = target[["date", "permno", "Return"]].rename(columns={"Return": "actual"})

        # -------------------------
        # 2. Iterate through each concept variable
        # -------------------------
        for concept_var in concept_vars:
            cons_sub = cons[["date", "permno", concept_var]].rename(columns={concept_var: "concept"})
            cons_sub["date"] = pd.to_datetime(cons_sub["date"])

            merged = pd.merge(cons_sub, target, on=["date", "permno"], how="inner")
            merged = merged.dropna(subset=["concept", "actual"])

            # Skip if insufficient variation
            if merged["concept"].nunique() < 10:
                print(f"  [Skip] λ={lam} — insufficient variation in {concept_var}.")
                continue

            # -------------------------
            # 3. Sort into deciles by concept variable
            # -------------------------
            decile_returns = []
            for date, df_date in merged.groupby("date"):
                if df_date["concept"].nunique() < 10:
                    continue
                try:
                    df_date["decile"] = pd.qcut(df_date["concept"], 10, labels=False, duplicates="drop") + 1
                except ValueError:
                    continue
                decile_means = df_date.groupby("decile")["actual"].mean()
                decile_returns.append(pd.DataFrame({"date": date, **decile_means.to_dict()}, index=[0]))

            if not decile_returns:
                continue

            port = pd.concat(decile_returns, ignore_index=True)
            port = port.sort_values("date").set_index("date")
            port.columns = [f"Decile{int(c)}" for c in port.columns]
            port = _restrict_period(port, start_dt, end_dt)

            # -------------------------
            # 4. Run GRS for each FF model
            # -------------------------
            for name in models_to_compare:
                if name not in std:
                    continue

                fac = _restrict_period(std[name]["factors"], start_dt, end_dt)
                rf = _restrict_period(std[name]["rf"], start_dt, end_dt)

                common_idx = port.index.intersection(fac.index).intersection(rf.index)
                if len(common_idx) < 24:
                    continue

                R = port.loc[common_idx]
                F = fac.loc[common_idx]
                rf_use = rf.loc[common_idx]

                pretty, res = _run_grs_named(R, name, F, rf_use)
                pretty["λ"] = lam
                pretty["Concept"] = concept_var
                summary_rows.append(pretty)
                details[f"{name} (λ={lam}, {concept_var})"] = res

    # -------------------------
    # 5. Summarize results
    # -------------------------
    if summary_rows:
        comparison_df = pd.DataFrame(summary_rows)
        ordered_cols = ["Concept", "λ"] + [c for c in comparison_df.columns if c not in ["Concept", "λ"]]
        comparison_df = comparison_df[ordered_cols]

        print("\n" + "=" * 90)
        print("Summary of GRS Tests across All Concepts")
        print("=" * 90)
        print(comparison_df.to_string(index=False, float_format="%.6f"))
    else:
        comparison_df = pd.DataFrame()
        print("No valid results produced.")

    return comparison_df, details


In [ ]:
# Compare model factors vs Fama-French factors
comp_df, details = compare_model_vs_ff_factors(
    horizon='12month',
    weight_lambda=[0.1, 0.5, 1.0],
    embedding_method='autoencoder',
    train_date='2020-01-01',
    start_date='1994-01-01',
    end_date='2020-01-01',
    data_dir='../data',
    portfolio_csv_path='../data/25_Portfolios_5x5.csv',
    portfolio_date_col='date',
    portfolio_yyyymm=True,
    portfolio_percent_to_decimal=True
)

Model Factors (one or many lambdas) vs Standard Factor Models

[Model] Running GRS for Model Factor (λ=0.1)
Running GRS test using model factors
Prediction horizon: 12month
Lambda: 0.1
Embedding method: autoencoder
Training date: 2020-01-01
--------------------------------------------------
1. Extracting model factors time series...
[INFO] CB-APM long–short factors constructed.
       shape=(314, 9), K=9
       Concepts: ['AnalystRevision', 'ChangeInRecommendation', 'ChForecastAccrual', 'EarningsForecastDisparity', 'FEPS']...
   Model factors shape: (314, 9)
   Factor names: ['AnalystRevision', 'ChangeInRecommendation', 'ChForecastAccrual', 'EarningsForecastDisparity', 'FEPS']...

2. Preparing user-specified portfolio returns...
   Portfolio returns shape: (313, 25)
   Common period: 1994-01-01 → 2019-12-01
   Final factors: (312, 9) | Final returns: (312, 25)

4. Running GRS test...

[Model] Running GRS for Model Factor (λ=0.5)
Running GRS test using model factors
Prediction horizon: 

In [44]:
comp_df, details = compare_model_vs_ff_factors(
    horizon='12month',
    weight_lambda=[0.1, 0.5, 1.0],
    embedding_method='autoencoder',
    train_date='2020-01-01',
    start_date='1994-01-01',
    end_date='2020-01-01',
    data_dir='../data',
    portfolio_csv_path='../data/25_Portfolios_ME_Prior_12_2.csv',
    portfolio_date_col='date',
    portfolio_yyyymm=True,
    portfolio_percent_to_decimal=True
)

Model Factors (one or many lambdas) vs Standard Factor Models

[Model] Running GRS for Model Factor (λ=0.1)
Running GRS test using model factors
Prediction horizon: 12month
Lambda: 0.1
Embedding method: autoencoder
Training date: 2020-01-01
--------------------------------------------------
1. Extracting model factors time series...
[INFO] CB-APM long–short factors constructed.
       shape=(314, 9), K=9
       Concepts: ['AnalystRevision', 'ChangeInRecommendation', 'ChForecastAccrual', 'EarningsForecastDisparity', 'FEPS']...
   Model factors shape: (314, 9)
   Factor names: ['AnalystRevision', 'ChangeInRecommendation', 'ChForecastAccrual', 'EarningsForecastDisparity', 'FEPS']...

2. Preparing user-specified portfolio returns...
   Portfolio returns shape: (313, 25)
   Common period: 1994-01-01 → 2019-12-01
   Final factors: (312, 9) | Final returns: (312, 25)

4. Running GRS test...

[Model] Running GRS for Model Factor (λ=0.5)
Running GRS test using model factors
Prediction horizon: 

In [45]:
comp_df, details = compare_model_vs_ff_factors(
    horizon='12month',
    weight_lambda=[0.1, 0.5, 1.0],
    embedding_method='autoencoder',
    train_date='2020-01-01',
    start_date='1994-01-01',
    end_date='2020-01-01',
    data_dir='../data',
    portfolio_csv_path='../data/30_Industry_Portfolios.csv',
    portfolio_date_col='date',
    portfolio_yyyymm=True,
    portfolio_percent_to_decimal=True
)

Model Factors (one or many lambdas) vs Standard Factor Models

[Model] Running GRS for Model Factor (λ=0.1)
Running GRS test using model factors
Prediction horizon: 12month
Lambda: 0.1
Embedding method: autoencoder
Training date: 2020-01-01
--------------------------------------------------
1. Extracting model factors time series...
[INFO] CB-APM long–short factors constructed.
       shape=(314, 9), K=9
       Concepts: ['AnalystRevision', 'ChangeInRecommendation', 'ChForecastAccrual', 'EarningsForecastDisparity', 'FEPS']...
   Model factors shape: (314, 9)
   Factor names: ['AnalystRevision', 'ChangeInRecommendation', 'ChForecastAccrual', 'EarningsForecastDisparity', 'FEPS']...

2. Preparing user-specified portfolio returns...
   Portfolio returns shape: (313, 30)
   Common period: 1994-01-01 → 2019-12-01
   Final factors: (312, 9) | Final returns: (312, 30)

4. Running GRS test...

[Model] Running GRS for Model Factor (λ=0.5)
Running GRS test using model factors
Prediction horizon: 

In [51]:
comp_df, details = compare_portfolio_vs_ff_factors(
    horizon="12month",
    weight_lambdas=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    data_dir="../data",
    results_dir="../results",
    start_date="1994-01-01",
    end_date="2020-01-01"
)

CB-APM Predicted Decile Portfolios vs Standard Factor Models

Summary of GRS Tests (Predicted Portfolios)
    Factor Model  GRS F-statistic  p-value  Mean Abs Alpha (Monthly)  Mean Abs Alpha (Annual)  RMS Alpha (Monthly)  RMS Alpha (Annual)  Number of Factors (K)  Sample Size (T)  Number of Assets (N)
    CAPM (λ=0.0)         1.988619 0.046592                  0.005160                 0.063780             0.005648            0.070084                      1               85                    10
     FF3 (λ=0.0)         1.891430 0.060301                  0.005406                 0.066892             0.005817            0.072247                      3               85                    10
Carhart4 (λ=0.0)         1.919169 0.056458                  0.006124                 0.076205             0.006520            0.081463                      4               85                    10
     FF5 (λ=0.0)         1.865708 0.064914                  0.005198                 0.064217             

In [52]:
comp_df, details = compare_concept_portfolio_vs_ff_factors(
    horizon="12month",
    weight_lambdas=[1.0],
    data_dir="../data",
    results_dir="../results",
    start_date="1994-01-01",
    end_date="2020-01-01"
)

CB-APM Concept-Sorted Decile Portfolios vs Standard Factor Models

Summary of GRS Tests across All Concepts
                       Concept        λ Factor Model  GRS F-statistic  p-value  Mean Abs Alpha (Monthly)  Mean Abs Alpha (Annual)  RMS Alpha (Monthly)  RMS Alpha (Annual)  Number of Factors (K)  Sample Size (T)  Number of Assets (N)
         EPS forecast revision 1.000000         CAPM         1.380516 0.206128                  0.004581                 0.056436             0.004735            0.058423                      1               85                    10
         EPS forecast revision 1.000000          FF3         1.369908 0.211733                  0.004997                 0.061668             0.005143            0.063598                      3               85                    10
         EPS forecast revision 1.000000     Carhart4         1.423652 0.187832                  0.005755                 0.071425             0.005975            0.074285                      4